Run cells in order from the **project root** or **`Baseline/`** (so `scripts/.env` is found).

1. `pip install openai sacrebleu tqdm`
2. Copy `scripts/.env.example` → `scripts/.env` and set `OPENROUTER_API_KEY`
3. Point `DATA_FILE` in cell 2 at your JSONL if not using the default `ende_dev_v2.jsonl`

Cell 6 runs `no_term`, `proper_term`, and `random_term` **in parallel** (one thread per mode).

Outputs go to `Baseline/openrouter_outputs/`.

In [10]:
# ============================================================
# Cell 1: Local setup — dependencies and OpenRouter client
# ============================================================
# Install once (from repo root or Baseline/):
#   pip install openai sacrebleu tqdm
#
# API key: scripts/.env  (see scripts/.env.example)

import os
from pathlib import Path

from openai import OpenAI


def find_repo_root() -> Path:
    """Locate project root by scripts/.env or scripts/.env.example."""
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "scripts" / ".env").is_file() or (path / "scripts" / ".env.example").is_file():
            return path
    raise FileNotFoundError(
        "Could not find repo root (expected scripts/.env). "
        "Open this notebook with the project folder as the working directory."
    )


def load_dotenv(path: Path) -> None:
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[7:].strip()
        key, sep, value = line.partition("=")
        if not sep:
            continue
        key = key.strip().lstrip("\ufeff")
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in "\"'":
            value = value[1:-1]
        os.environ.setdefault(key, value)


REPO_ROOT = find_repo_root()
SCRIPTS_DIR = REPO_ROOT / "scripts"
ENV_FILE = SCRIPTS_DIR / ".env"

load_dotenv(ENV_FILE)

api_key = os.environ.get("OPENROUTER_API_KEY")
if not api_key:
    raise EnvironmentError(
        f"OPENROUTER_API_KEY not set. Add it to {ENV_FILE} "
        "(copy from scripts/.env.example)."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

OPENAI_MODEL = "openai/gpt-4o-mini"

print("Repo root:", REPO_ROOT)
print("Env file:", ENV_FILE if ENV_FILE.is_file() else "(missing — using existing env vars)")
print("OpenRouter client ready")

Repo root: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation
Env file: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\scripts\.env
OpenRouter client ready


In [13]:
# ============================================================
# Cell 2: Paths and settings (repo-relative)
# ============================================================

BASELINE_DIR = REPO_ROOT / "Baseline"

# Default: Baseline/ende_dev_v2.jsonl — override with an absolute path if needed
DATA_FILE = BASELINE_DIR / "ende_dev_v2.jsonl"
OUTPUT_DIR = BASELINE_DIR / "openrouter_outputs"

# Use a small number first for testing, then set to None for full run.
MAX_SAMPLES = None   # example: 10, 100, or None

# Modes to run:
# - no_term: translate without terminology
# - proper_term: use proper_terms
# - random_term: use random_terms, excluding proper_terms
MODES = ["no_term", "proper_term", "random_term"]

TARGET_LANG = "German"
OUTPUT_TAG = "deu"
REF_FIELD = "de"

TOP_TERM_PREVIEW = 5
TOP_SAMPLE_PREVIEW = 2

DATA_STEM = Path(DATA_FILE).stem

if not Path(DATA_FILE).is_file():
    raise FileNotFoundError(f"File not found: {DATA_FILE}")

print("Using data file:", DATA_FILE)
print("Output directory:", OUTPUT_DIR)

Using data file: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline\ende_dev_v2.jsonl
Output directory: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline\openrouter_outputs


In [15]:
# ============================================================
# Cell 3: Helper functions
# ============================================================

import json
import re
import time
import sacrebleu

from typing import List, Dict, Optional
from collections import defaultdict, Counter
from tqdm import tqdm


def load_jsonl(path: str) -> List[Dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def save_jsonl(path: str, records: List[Dict]):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def strip_output_tags(text: str) -> str:
    """Remove translation output tags such as <deu>...</deu>."""
    if not isinstance(text, str):
        return text
    return re.sub(r"</?(deu|de|ger|german)>", "", text, flags=re.IGNORECASE).strip()


def compute_bleu_chrf(hyps: List[str], refs: List[str]) -> Dict:
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return {
        "bleu": bleu.score,
        "chrf": chrf.score,
    }


def _normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())


def _count_term_occurrences(text: str, term: str) -> int:
    text_norm = _normalize_text(text)
    term_norm = _normalize_text(term)
    pattern = r"\b" + re.escape(term_norm) + r"\b"
    return len(re.findall(pattern, text_norm))


def terminology_accuracy_advanced(
    preds: List[str],
    samples: List[Dict],
    mode: str = "proper_term"
) -> Dict:
    term_ratios = {}
    total_terms = 0

    for pred, sample in zip(preds, samples):
        if mode == "proper_term":
            terms = (sample.get("proper_terms") or {}).copy()
        elif mode == "random_term":
            terms = (sample.get("random_terms") or {}).copy()
            for key in (sample.get("proper_terms") or {}).keys():
                terms.pop(key, None)
        else:
            terms = {}

        source_text = sample.get("en", "")

        for src, tgt in terms.items():
            total_terms += 1

            src_count = _count_term_occurrences(source_text, src)
            if src_count == 0:
                src_count = 1

            tgt_count = _count_term_occurrences(pred, tgt)
            ratio = min(tgt_count / src_count, 1.0)

            term_ratios[src] = ratio

    avg_accuracy = (
        sum(term_ratios.values()) / len(term_ratios) * 100
        if term_ratios
        else None
    )

    return {
        "total_terms": total_terms,
        "avg_ratio_pct": avg_accuracy,
        "per_term_ratios": term_ratios,
    }


def terminology_consistency_advanced(
    preds: List[str],
    samples: List[Dict],
    mode: str = "proper_term"
) -> Dict:
    term_to_candidates = defaultdict(list)

    for pred, sample in zip(preds, samples):
        if mode == "proper_term":
            terms = (sample.get("proper_terms") or {}).copy()
        elif mode == "random_term":
            terms = (sample.get("random_terms") or {}).copy()
            for key in (sample.get("proper_terms") or {}).keys():
                terms.pop(key, None)
        else:
            terms = {}

        for src, tgt in terms.items():
            if str(tgt).lower() in str(pred).lower():
                term_to_candidates[src].append(tgt)
            else:
                term_to_candidates[src].append("<MISSING>")

    pseudo_references = {}

    for src, candidates in term_to_candidates.items():
        counter = Counter(candidates)
        pseudo_references[src] = counter.most_common(1)[0][0]

    per_term_consistency = {}
    macro_scores = []
    weighted_scores = []

    for src, candidates in term_to_candidates.items():
        pseudo_ref = pseudo_references[src]
        matches = sum(1 for c in candidates if c == pseudo_ref)
        consistency = matches / len(candidates) if candidates else 0.0

        per_term_consistency[src] = {
            "occ": len(candidates),
            "pseudo_ref": pseudo_ref,
            "matches": matches,
            "consistency": consistency,
        }

        macro_scores.append(consistency)
        weighted_scores.extend([consistency] * len(candidates))

    return {
        "per_term": per_term_consistency,
        "macro_avg_consistency": (
            sum(macro_scores) / len(macro_scores)
            if macro_scores
            else None
        ),
        "weighted_avg_consistency": (
            sum(weighted_scores) / len(weighted_scores)
            if weighted_scores
            else None
        ),
    }


def _fmt_metric(value, digits=2):
    if value is None:
        return "N/A"
    return f"{value:.{digits}f}"


def _preview_term_stats(term_stats, limit=5):
    if not term_stats:
        return []

    items = sorted(
        term_stats.items(),
        key=lambda item: (-item[1].get("occ", 0), item[0])
    )

    return items[:limit]

In [16]:
# ============================================================
# Cell 4: OpenRouter English → German translation function
# ============================================================

def translate_sample(
    sample_en: str,
    terminology: Optional[Dict[str, str]] = None,
    target_lang: str = TARGET_LANG,
    output_tag: str = OUTPUT_TAG,
    max_new_tokens: int = 256,
    retries: int = 3,
    sleep_seconds: int = 2,
) -> str:
    term_block = ""

    if terminology:
        term_block = "Terminology:\n"
        for src, tgt in terminology.items():
            term_block += f"{src} -> {tgt}\n"
        term_block += "\n"

    prompt = f"""
Translate the English text to {target_lang}.

Rules:
1. Output only in this format: <{output_tag}> ... </{output_tag}>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.
4. Translate only from English to German.

{term_block}
Input:
<en> {sample_en} </en>
"""

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a translation assistant. You only translate English to German."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                max_tokens=max_new_tokens,
                temperature=0,
            )

            return response.choices[0].message.content.strip()

        except Exception as e:
            last_error = e
            print(f"API error on attempt {attempt}/{retries}: {e}")
            if attempt < retries:
                time.sleep(sleep_seconds)

    raise RuntimeError(f"Translation failed after {retries} attempts: {last_error}")

In [17]:
# ============================================================
# Cell 5: Load JSONL dataset
# ============================================================

samples = load_jsonl(str(DATA_FILE))

if MAX_SAMPLES is not None:
    samples = samples[:MAX_SAMPLES]

print(f"Loaded samples: {len(samples)}")
print("First sample keys:", samples[0].keys())

if "en" not in samples[0]:
    raise KeyError("Expected source field 'en' was not found.")

if REF_FIELD not in samples[0]:
    print(f"Warning: reference field '{REF_FIELD}' not found. BLEU/chrF cannot be computed.")
else:
    print(f"Reference field found: '{REF_FIELD}'")

Loaded samples: 2000
First sample keys: dict_keys(['en', 'de', 'proper_terms', 'random_terms'])
Reference field found: 'de'


In [18]:
# ============================================================
# Cell 6: Run EN → DE translation and evaluation (parallel modes)
# ============================================================

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

_print_lock = threading.Lock()


def _log(*args, **kwargs):
    with _print_lock:
        print(*args, **kwargs)


def terminology_for_mode(sample: Dict, mode: str) -> Optional[Dict[str, str]]:
    if mode == "no_term":
        return None
    if mode == "proper_term":
        return sample.get("proper_terms") or None
    if mode == "random_term":
        terminology = (sample.get("random_terms") or {}).copy()
        for key in (sample.get("proper_terms") or {}).keys():
            terminology.pop(key, None)
        return terminology or None
    return None


def term_eval_mode(mode: str) -> str:
    if mode == "proper_term":
        return "proper_term"
    if mode == "random_term":
        return "random_term"
    return "no_term"


def run_mode_evaluation(
    mode: str,
    samples: List[Dict],
    base_output_dir: Path,
    tqdm_position: int,
) -> Dict:
    _log(f"\n--- Mode: {mode} (started) ---")

    preds = []
    records = []

    for sample in tqdm(
        samples,
        desc=f"{DATA_STEM} - {mode}",
        position=tqdm_position,
        leave=True,
    ):
        pred = translate_sample(
            sample_en=sample.get("en", ""),
            terminology=terminology_for_mode(sample, mode),
            target_lang=TARGET_LANG,
            output_tag=OUTPUT_TAG,
        )
        preds.append(pred)

        record = sample.copy()
        record[f"prediction_{mode}"] = pred
        record[f"prediction_{mode}_clean"] = strip_output_tags(pred)
        records.append(record)

    clean_preds = [strip_output_tags(p) for p in preds]
    output_jsonl = base_output_dir / f"{DATA_STEM}_{mode}_predictions.jsonl"
    save_jsonl(str(output_jsonl), records)
    _log(f"[{mode}] Saved predictions to: {output_jsonl}")

    metrics = {}

    if REF_FIELD in samples[0]:
        refs = [sample.get(REF_FIELD, "") for sample in samples]
        metrics.update(compute_bleu_chrf(clean_preds, refs))

        term_mode = term_eval_mode(mode)
        term_acc = terminology_accuracy_advanced(clean_preds, samples, mode=term_mode)
        term_cons = terminology_consistency_advanced(clean_preds, samples, mode=term_mode)

        metrics["terminology_accuracy"] = term_acc
        metrics["terminology_consistency"] = term_cons

        _log(f"[{mode}] BLEU: {_fmt_metric(metrics['bleu'])}")
        _log(f"[{mode}] chrF2++: {_fmt_metric(metrics['chrf'])}")
        _log(
            f"[{mode}] Terminology accuracy ratio %: "
            f"{_fmt_metric(term_acc.get('avg_ratio_pct'))}"
        )
        _log(f"[{mode}] Terminology terms counted: {term_acc.get('total_terms', 0)}")
        _log(
            f"[{mode}] Macro-avg consistency: "
            f"{_fmt_metric(term_cons.get('macro_avg_consistency'))}"
        )
        _log(
            f"[{mode}] Weighted-avg consistency: "
            f"{_fmt_metric(term_cons.get('weighted_avg_consistency'))}"
        )

        if term_acc.get("per_term_ratios"):
            _log(f"[{mode}] Top terminology accuracy terms:")
            preview_data = {
                term: {"occ": 1, "ratio": ratio}
                for term, ratio in term_acc["per_term_ratios"].items()
            }
            for term, ratio_info in _preview_term_stats(preview_data, limit=TOP_TERM_PREVIEW):
                _log(f"[{mode}]   - {term}: ratio={_fmt_metric(ratio_info['ratio'])}")
        else:
            _log(f"[{mode}] Top terminology accuracy terms: N/A")

        per_term_consistency = term_cons.get("per_term") or {}
        if per_term_consistency:
            _log(f"[{mode}] Top terminology consistency terms:")
            for term, stats in _preview_term_stats(per_term_consistency, limit=TOP_TERM_PREVIEW):
                _log(
                    f"[{mode}]   - {term}: occ={stats.get('occ', 0)}, "
                    f"pseudo_ref={stats.get('pseudo_ref')}, "
                    f"consistency={_fmt_metric(stats.get('consistency'))}"
                )
        else:
            _log(f"[{mode}] Top terminology consistency terms: N/A")
    else:
        _log(f"[{mode}] Skipping BLEU/chrF — reference field '{REF_FIELD}' not found.")

    _log(f"[{mode}] Sample previews:")
    for idx, (sample, pred) in enumerate(
        list(zip(samples, preds))[:TOP_SAMPLE_PREVIEW],
        start=1,
    ):
        ref = sample.get(REF_FIELD)
        _log(f"[{mode}]   [{idx}] EN: {sample.get('en', '').strip()}")
        _log(f"[{mode}]       PRED: {pred.strip()}")
        if isinstance(ref, str):
            _log(f"[{mode}]       REF : {ref.strip()}")
        else:
            _log(f"[{mode}]       REF : {ref}")

    _log(f"--- Mode: {mode} (finished) ---")
    return {
        "predictions_file": str(output_jsonl),
        "metrics": metrics,
    }


base_output_dir = Path(OUTPUT_DIR)
base_output_dir.mkdir(parents=True, exist_ok=True)

print(f"\n=== File: {DATA_FILE}")
print(f"=== Translation direction: English → German")
print(f"=== Samples: {len(samples)}")
print(f"=== MAX_SAMPLES: {MAX_SAMPLES}")
print(f"=== Model: {OPENAI_MODEL}")
print(f"=== Modes (parallel, max_workers={len(MODES)}): {MODES}")

all_results = {}

with ThreadPoolExecutor(max_workers=len(MODES)) as executor:
    futures = {
        executor.submit(
            run_mode_evaluation,
            mode,
            samples,
            base_output_dir,
            MODES.index(mode),
        ): mode
        for mode in MODES
    }
    for future in as_completed(futures):
        mode = futures[future]
        all_results[mode] = future.result()

metrics_path = base_output_dir / f"{DATA_STEM}_metrics_summary.json"

with open(str(metrics_path), "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\nEvaluation finished.")
print("Saved metrics summary to:", metrics_path)


=== File: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline\ende_dev_v2.jsonl
=== Translation direction: English → German
=== Samples: 2000
=== MAX_SAMPLES: None
=== Model: openai/gpt-4o-mini
=== Modes (parallel, max_workers=3): ['no_term', 'proper_term', 'random_term']

--- Mode: no_term (started) ---

--- Mode: proper_term (started) ---


ende_dev_v2 - no_term:   0%|          | 0/2000 [00:00<?, ?it/s]


--- Mode: random_term (started) ---



ende_dev_v2 - no_term:   0%|          | 1/2000 [00:02<1:15:51,  2.28s/it]

ende_dev_v2 - no_term:   1%|          | 14/2000 [00:36<1:12:23,  2.19s/it]

ende_dev_v2 - no_term:   1%|          | 19/2000 [00:49<1:16:33,  2.32s/it]

ende_dev_v2 - no_term:   1%|▏         | 27/2000 [01:05<1:10:02,  2.13s/it]

ende_dev_v2 - no_term:   1%|▏         | 28/2000 [01:11<1:43:36,  3.15s/it]

ende_dev_v2 - no_term:   1%|▏         | 29/2000 [01:14<1:39:38,  3.03s/it]

ende_dev_v2 - no_term:   2%|▏         | 38/2000 [01:37<1:18:37,  2.40s/it]

ende_dev_v2 - no_term:   2%|▏         | 47/2000 [02:00<1:04:23,  1.98s/it]

ende_dev_v2 - no_term:   2%|▏         | 48/2000 [02:04<1:30:15,  2.77s/it]

ende_dev_v2 - no_term:   2%|▏         | 49/2000 [02:07<1:33:00,  2.86s/it]

ende_dev_v2 - no_term:   2%|▎         | 50/2000 [02:11<1:40:22,  3.09s/it]

ende_dev_v2 - no_term:   3%|▎         | 51/2000 [02:16<1:54:46,  3.53s/it]

ende_dev_v2 - no_term:   3%|▎         | 55/2000 [02:25<1:27:46,  2.71s/it]


ende_dev_v2

[no_term] Saved predictions to: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline\openrouter_outputs\ende_dev_v2_no_term_predictions.jsonl


[no_term] BLEU: 43.78
[no_term] chrF2++: 68.93
[no_term] Terminology accuracy ratio %: N/A
[no_term] Terminology terms counted: 0
[no_term] Macro-avg consistency: N/A
[no_term] Weighted-avg consistency: N/A
[no_term] Top terminology accuracy terms: N/A
[no_term] Top terminology consistency terms: N/A
[no_term] Sample previews:
[no_term]   [1] EN: The status of each individual space can be seen from the color code on the upper left corner.
[no_term]       PRED: <deu> Der Status jedes einzelnen Raums ist an dem Farbcodes in der oberen linken Ecke zu erkennen. </deu>
[no_term]       REF : Der Farbcode oben links gibt den Status des betreffenden Space an.
[no_term]   [2] EN: This service describes the deployed (run-time) state of SAP HANA database artifacts, for example: tables, views, or procedures, which have been created or adjusted by the SAP Integrated Development Environment (WebIDE) editors as a family of consistent design-time artifacts for all key SAP HANA platform database featur

















ende_dev_v2 - proper_term: 100%|██████████| 2000/2000 [1:17:06<00:00,  2.31s/it]


[proper_term] Saved predictions to: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline\openrouter_outputs\ende_dev_v2_proper_term_predictions.jsonl
[proper_term] BLEU: 50.54
[proper_term] chrF2++: 75.19
[proper_term] Terminology accuracy ratio %: 86.56
[proper_term] Terminology terms counted: 2754
[proper_term] Macro-avg consistency: 0.96
[proper_term] Weighted-avg consistency: 0.88
[proper_term] Top terminology accuracy terms:
[proper_term]   - <section>: ratio=0.00
[proper_term]   - <sectiondiv>: ratio=0.00
[proper_term]   - ABAP Manageable Trigger Namespace: ratio=1.00
[proper_term]   - API endpoint: ratio=1.00
[proper_term]   - APL: ratio=1.00
[proper_term] Top terminology consistency terms:
[proper_term]   - space: occ=90, pseudo_ref=Space, consistency=0.90
[proper_term]   - create: occ=80, pseudo_ref=erstellen, consistency=0.84
[proper_term]   - provider: occ=42, pseudo_ref=Provider, consistency=0.86
[proper_term]   - data provider: 












ende_dev_v2 - random_term: 100%|██████████| 2000/2000 [1:17:29<00:00,  2.32s/it]


[random_term] Saved predictions to: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline\openrouter_outputs\ende_dev_v2_random_term_predictions.jsonl
[random_term] BLEU: 45.06
[random_term] chrF2++: 70.92
[random_term] Terminology accuracy ratio %: 88.94
[random_term] Terminology terms counted: 2875
[random_term] Macro-avg consistency: 0.90
[random_term] Weighted-avg consistency: 0.79
[random_term] Top terminology accuracy terms:
[random_term]   - +: ratio=0.00
[random_term]   - Accept: ratio=1.00
[random_term]   - Access: ratio=1.00
[random_term]   - Accessing: ratio=1.00
[random_term]   - Account: ratio=1.00
[random_term] Top terminology consistency terms:
[random_term]   - Select: occ=90, pseudo_ref=Wählen, consistency=0.94
[random_term]   - Click: occ=60, pseudo_ref=Klicken, consistency=0.70
[random_term]   - click: occ=33, pseudo_ref=klicken, consistency=0.48
[random_term]   - Create: occ=32, pseudo_ref=Erstellen, consistency=0.50
[rand